# 11. Token-budget probe run (200M baseline, ~1B tokens)

Goal: train a single **200M dense decoder-only baseline** on FineWeb-Edu for a **~1B token** budget to probe the loss trajectory before committing to the full ladder. **Train + checkpoint only — no inline evals** (evals run separately later against the saved Drive checkpoints).

- Model: baseline only, ~203.6M params (d_model 1024, n_layers 12, n_heads 16, d_ff 4096)
- Hyperparameters: blessed 30M values, scaled proportionally (AdamW 0.9/0.95, wd 0.1, eps 1e-8, cosine + warmup, bf16, effective batch 65,536 tokens)
- Dataset: `fineweb_edu` sample-10BT, block_size 1024, existing hash split
- Hardware: Colab A100 40GB
- Checkpoints: every ~150M tokens (7 total) written **directly to Google Drive**

Flow: sync repo -> mount Drive -> GPU check -> print config/schedule/cost -> VRAM probe -> **confirmation gate** -> launch. Config: `configs/fineweb_200m_budget.yaml`.

In [ ]:
import os
import sys
import time
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/AtinChing/AttnResGPT-next-scale.git"
REPO_TARGET = Path("/content/AttnResGPT-next-scale")
DRIVE_ARTIFACT_FOLDER = "AttnResGPT-next-scale-artifacts"


def is_repo_root(path: Path) -> bool:
    return (path / "src" / "training" / "train.py").exists() and (path / "requirements.txt").exists()


def discover_repo() -> Path:
    for candidate in [REPO_TARGET, Path.cwd(), Path.cwd().parent]:
        if is_repo_root(candidate):
            return candidate.resolve()
    if not REPO_TARGET.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_TARGET)], check=True)
    return REPO_TARGET.resolve()


REPO = discover_repo()
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# W&B: enabled with mode=auto (online if WANDB_API_KEY set, else offline). Paste your key below if desired.
# os.environ["WANDB_API_KEY"] = "<your-key>"
print("repo:", REPO)
print("WANDB_API_KEY set =", bool(os.environ.get("WANDB_API_KEY")))

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_ARTIFACT_FOLDER
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("drive root:", DRIVE_ROOT)
print("checkpoints will be written directly under:", DRIVE_ROOT / "checkpoints")

In [ ]:
import torch

print("cuda_available =", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("device_name =", props.name)
    print("total_vram_gib =", round(props.total_memory / (1024 ** 3), 1))
    print("bf16_supported =", torch.cuda.is_bf16_supported())
else:
    print("WARNING: no CUDA device. This notebook targets a Colab A100 40GB.")

## Config, token budget, checkpoint schedule, and cost (printed before launch)

The 200M config and horizon live in `configs/fineweb_200m_budget.yaml`. The cell below loads it, reports the exact param count and token/checkpoint schedule, and prints a rough A100 wall-clock / compute-unit estimate.

In [ ]:
from src.utils.config import load_config
from src.models.attnres import build_model
from src.utils.runtime import count_parameters

CONFIG_PATH = "configs/fineweb_200m_budget.yaml"
EFFECTIVE_BATCH_TOKENS = 65_536

cfg = load_config(CONFIG_PATH)
param_total = count_parameters(build_model(cfg.model))["total"]

eff_tokens = cfg.data.batch_size * cfg.training.grad_accum_steps * cfg.data.block_size
total_tokens = cfg.training.max_steps * eff_tokens
ckpt_steps = list(range(cfg.training.checkpoint_interval, cfg.training.max_steps + 1, cfg.training.checkpoint_interval))
if cfg.training.max_steps not in ckpt_steps:
    ckpt_steps.append(cfg.training.max_steps)

print("=== 200M token-budget probe: pre-launch report ===")
print(f"model         : baseline dense decoder-only")
print(f"  d_model={cfg.model.d_model} n_layers={cfg.model.n_layers} n_heads={cfg.model.n_heads} d_ff={cfg.model.d_ff}")
print(f"  param_count = {param_total:,} (~{param_total/1e6:.1f}M)")
print(f"optimizer     : AdamW betas=({cfg.training.beta1},{cfg.training.beta2}) eps={cfg.training.adam_eps} wd={cfg.training.weight_decay}")
print(f"schedule      : cosine + warmup | peak={cfg.training.learning_rate} min={cfg.training.min_lr} warmup={cfg.training.warmup_steps}")
print(f"precision     : {'bf16' if cfg.training.mixed_precision else 'fp32'} ({cfg.training.amp_dtype})")
print(f"effective batch: {cfg.data.batch_size} micro x {cfg.training.grad_accum_steps} accum x {cfg.data.block_size} ctx = {eff_tokens:,} tokens/step")
print(f"horizon       : max_steps={cfg.training.max_steps} -> {total_tokens:,} tokens ({total_tokens/1e9:.3f}B)")
print(f"eval          : eval_interval={cfg.training.eval_interval} (0 = NO inline/final eval; train + checkpoint only)")
print()
print(f"checkpoints   : every {cfg.training.checkpoint_interval} steps (~{cfg.training.checkpoint_interval*eff_tokens/1e6:.1f}M tokens), keep_last_k={cfg.logging.keep_last_k_checkpoints}")
for s in ckpt_steps:
    print(f"    step {s:>6}  ~{s*eff_tokens/1e6:6.0f}M tokens")
print(f"  total checkpoints: {len(ckpt_steps)} (written directly to Drive)")
print()
# rough A100 cost estimate
train_flops = 6 * param_total * total_tokens
A100_BF16_PEAK_FLOPS = 312e12
for mfu, label in [(0.30, "conservative"), (0.45, "optimistic")]:
    eff_flops = mfu * A100_BF16_PEAK_FLOPS
    hours = train_flops / eff_flops / 3600
    print(f"est wall-clock ({label}, MFU {mfu:.0%}): {hours:.1f} h  (~{total_tokens/ (hours*3600):,.0f} tok/s)")
print("est Colab A100 cost: ~12 CU/h => roughly 25-55 CU for this run (approximate; depends on live rate + throughput).")

## VRAM probe: pick the largest micro-batch that fits (effective batch stays 65,536 tokens)

Finds the largest micro-batch that fits under 90% of A100 VRAM via a real fwd+bwd+optimizer step on synthetic data, then sets `grad_accum` to hold the effective batch at 64 sequences. Prefers micro-batch 64 (accum 1), then 32 (accum 2), then 16 (accum 4).

In [ ]:
from torch.optim import AdamW
from src.metrics.norms import language_model_loss
from src.utils.runtime import amp_dtype_from_string

EFFECTIVE_BATCH_SEQUENCES = EFFECTIVE_BATCH_TOKENS // cfg.data.block_size  # 64
CANDIDATE_MICRO_BATCHES = [64, 32, 16]   # must divide EFFECTIVE_BATCH_SEQUENCES
VRAM_HEADROOM = 0.90

# Defaults if no CUDA (e.g. dry run): the config's own values.
MICRO_BATCH = cfg.data.batch_size
GRAD_ACCUM = cfg.training.grad_accum_steps


def _peak_mib_for_micro_batch(micro_batch: int, steps: int = 3) -> float:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    model = build_model(cfg.model).to("cuda")
    opt = AdamW(model.parameters(), lr=cfg.training.learning_rate,
                betas=(cfg.training.beta1, cfg.training.beta2),
                eps=cfg.training.adam_eps, weight_decay=cfg.training.weight_decay)
    amp_dtype = amp_dtype_from_string(cfg.training.amp_dtype)
    model.train()
    for _ in range(steps):
        x = torch.randint(0, cfg.model.vocab_size, (micro_batch, cfg.data.block_size), device="cuda")
        y = torch.randint(0, cfg.model.vocab_size, (micro_batch, cfg.data.block_size), device="cuda")
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=cfg.training.mixed_precision):
            logits, _ = model(x, return_aux=False)
            loss = language_model_loss(logits, y)
        loss.backward()
        opt.step()
    peak = torch.cuda.max_memory_allocated() / (1024 ** 2)
    del model, opt
    torch.cuda.empty_cache()
    return peak


if torch.cuda.is_available():
    total_mib = torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)
    budget = VRAM_HEADROOM * total_mib
    print(f"VRAM budget: {budget:,.0f} / {total_mib:,.0f} MiB @ {VRAM_HEADROOM:.0%}")
    chosen = None
    for mb in sorted(CANDIDATE_MICRO_BATCHES, reverse=True):
        if EFFECTIVE_BATCH_SEQUENCES % mb != 0:
            continue
        try:
            peak = _peak_mib_for_micro_batch(mb)
        except torch.cuda.OutOfMemoryError:
            print(f"micro_batch={mb:>3}: OOM")
            torch.cuda.empty_cache()
            continue
        fits = peak <= budget
        print(f"micro_batch={mb:>3}: peak {peak:8,.0f} MiB [{'OK' if fits else 'over budget'}]")
        if fits and chosen is None:
            chosen = mb
    if chosen is not None:
        MICRO_BATCH = chosen
        GRAD_ACCUM = EFFECTIVE_BATCH_SEQUENCES // chosen
    print(f"\nchosen micro_batch={MICRO_BATCH}, grad_accum={GRAD_ACCUM} "
          f"-> {MICRO_BATCH*GRAD_ACCUM} seqs = {MICRO_BATCH*GRAD_ACCUM*cfg.data.block_size:,} tokens/step")
else:
    print(f"No CUDA; using config defaults micro_batch={MICRO_BATCH}, grad_accum={GRAD_ACCUM}")

## Confirmation gate (must be True to launch)

Review the config, schedule, cost, and chosen micro-batch above. The next cell assembles the final overrides (including `logging.output_root` pointed at Drive so checkpoints save directly to Drive). Training launches only when `CONFIRM_LAUNCH = True`.

In [ ]:
CONFIRM_LAUNCH = False

from src.utils.logging import build_run_identity

# Use the canonical identity so checkpoints and W&B resume the same run after a disconnect.
RUN_NAME = build_run_identity(cfg).run_name

# output_root -> Drive so every checkpoint torch.save writes straight to Drive.
OVERRIDES = [
    f"logging.output_root={DRIVE_ROOT}",
    f"data.batch_size={MICRO_BATCH}",
    f"data.eval_batch_size={MICRO_BATCH}",
    f"training.grad_accum_steps={GRAD_ACCUM}",
    f"logging.wandb.run_name={RUN_NAME}",
]

print("final launch plan:")
print("  config       :", CONFIG_PATH)
print("  run_name     :", RUN_NAME)
print("  output_root  :", DRIVE_ROOT, "(checkpoints -> Drive as they save)")
print("  overrides    :", OVERRIDES)
print("  CONFIRM_LAUNCH =", CONFIRM_LAUNCH)
print("\nSet CONFIRM_LAUNCH = True, then run the next cell to start the ~1B token run.")

In [ ]:
if not CONFIRM_LAUNCH:
    raise RuntimeError("Set CONFIRM_LAUNCH = True in the previous cell after reviewing the plan.")

from src.training.train import train_from_config

launch_cfg = load_config(CONFIG_PATH, overrides=OVERRIDES)
assert launch_cfg.training.eval_interval == 0, "eval must be disabled for this train-only run"
assert (launch_cfg.data.batch_size * launch_cfg.training.grad_accum_steps * launch_cfg.data.block_size
        == EFFECTIVE_BATCH_TOKENS), "effective batch drifted from 65,536 tokens"

launch_start = time.time()
summary = train_from_config(launch_cfg)
print(f"\nTraining finished in {(time.time() - launch_start)/3600:.2f} h")
print("checkpoints on Drive:", DRIVE_ROOT / "checkpoints" / summary["run_name"])
print("run summary:", {k: summary.get(k) for k in [
    "run_name", "parameter_count_total", "precision_mode",
    "effective_batch_tokens", "tokens_seen", "checkpoint_path"]})
_TRAINING_COMPLETED = True

## Resume after a disconnect

If Colab dropped mid-run, re-run cells 1-9 (sync, Drive mount, config, probe, plan) on the fresh runtime, then set `CONFIRM_RESUME = True` and run the cell below. It picks the **latest checkpoint on Drive** and continues from that step with the identical config (same effective batch, cosine schedule, optimizer/scheduler/RNG state are restored from the checkpoint). Do **not** also run the fresh-launch cell in the same pass.

In [ ]:
CONFIRM_RESUME = False

if not CONFIRM_RESUME:
    print("Resume skipped (CONFIRM_RESUME=False). Set it True after a disconnect to continue from the latest Drive checkpoint.")
else:
    from src.training.train import train_from_config

    resume_dir = DRIVE_ROOT / "checkpoints" / RUN_NAME
    ckpts = sorted(resume_dir.glob("step_*.pt"))
    if not ckpts:
        raise FileNotFoundError(f"No checkpoints found on Drive at {resume_dir}")
    latest_ckpt = ckpts[-1]
    print("resuming from:", latest_ckpt)

    resume_overrides = OVERRIDES + [f"training.resume_from={latest_ckpt}"]
    resume_cfg = load_config(CONFIG_PATH, overrides=resume_overrides)
    assert resume_cfg.training.eval_interval == 0
    assert (resume_cfg.data.batch_size * resume_cfg.training.grad_accum_steps * resume_cfg.data.block_size
            == EFFECTIVE_BATCH_TOKENS), "effective batch drifted from 65,536 tokens"

    resume_start = time.time()
    summary = train_from_config(resume_cfg)
    print(f"\nResumed training finished in {(time.time() - resume_start)/3600:.2f} h")
    print("checkpoints on Drive:", resume_dir)
    print("run summary:", {k: summary.get(k) for k in [
        "run_name", "tokens_seen", "checkpoint_path"]})
    _TRAINING_COMPLETED = True

## End the Colab session on completion

Runs after training (fresh or resumed) finishes, releasing the A100 so idle compute units aren't burned. It only unassigns if a training run actually completed in this session (so a partial "Run all" without confirmation won't kill the runtime). Set `UNASSIGN_ON_COMPLETE = False` to keep the session alive.

In [ ]:
UNASSIGN_ON_COMPLETE = True

training_completed = globals().get("_TRAINING_COMPLETED", False)

if UNASSIGN_ON_COMPLETE and training_completed:
    print("Training complete - ending Colab session to free the A100.")
    from google.colab import runtime
    runtime.unassign()
else:
    print(f"Not unassigning (UNASSIGN_ON_COMPLETE={UNASSIGN_ON_COMPLETE}, training_completed={training_completed}).")